## Set up 

In [1]:
# %% Libraries
import os
import scanpy as sc
#import scvi
import numpy as np
from pathlib import Path
import pandas as pd
import anndata as ad
import rapids_singlecell as rsc


# %% Setting Paths
MAIN_DIR_NAME = "cosmx_gray"
MAIN_DIR = next(p for p in Path.cwd().parents if (p / MAIN_DIR_NAME).exists()) / MAIN_DIR_NAME
os.chdir(MAIN_DIR)

# %% Setting Seed
SEED_VALUE = 42
# set NumPy RNG for consistency
np.random.seed(SEED_VALUE)

# %% object versions
CUR_OBJ_V = 'v8'
CUR_OBJ_PATH = MAIN_DIR / 'data' / 'comb' / 'h5ad' / f'comb_{CUR_OBJ_V}.h5ad'

# %% Load the unintegrated object
comb = sc.read_h5ad(CUR_OBJ_PATH)

# PseudoBulk

aggregate_counts function

In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import issparse

def aggregate_counts(
    adata,
    celltype_col: str,
    samples_col: str,
    celltype_value: str,
    layer: str,
    shape: str = "samples-genes",  # "samples-genes" or "genes-samples"
):
    """
    Filter AnnData to one cell type and return a DataFrame of summed counts.

    Parameters
    ----------
    shape:
      - "samples-genes": rows = samples, columns = genes
      - "genes-samples": rows = genes, columns = samples
    """
    if shape not in {"samples-genes", "genes-samples"}:
        raise ValueError("shape must be one of: 'samples-genes', 'genes-samples'")

    # Filter to chosen cell type
    subset = adata[adata.obs[celltype_col] == celltype_value].copy()

    # Sample labels (one per cell)
    sample_ids = subset.obs[samples_col].astype(str).to_numpy()

    # Extract matrix from layer
    if layer not in subset.layers:
        raise ValueError(f"layer {layer!r} not found in adata.layers")

    X = subset.layers[layer]
    X = X.toarray() if issparse(X) else np.asarray(X)

    # cells × genes dataframe (index = sample id per cell)
    df_cells_genes = pd.DataFrame(X, index=sample_ids, columns=subset.var_names)

    # samples × genes
    df_samples_genes = df_cells_genes.groupby(level=0).sum()

    if shape == "samples-genes":
        return df_samples_genes
    else:  # "genes-samples"
        return df_samples_genes.T


## Generate pb counts

In [6]:
celltypes = comb.obs['ann_lvl_3'].unique().tolist()

power_path = MAIN_DIR / 'results' / 'comb' / 'power' 

for ct in celltypes:
    print(f"Processing cell type: {ct}")
    pb_df = aggregate_counts(
        comb,
        celltype_col='ann_lvl_3',
        samples_col='donor',
        celltype_value=ct,
        layer='counts',
        shape='genes-samples',
        #shape='samples-genes'
    )

    # save pseudo-bulk dataframe
    out_dir = power_path / ct / f'pb_counts.csv'
    out_dir.parent.mkdir(parents=True, exist_ok=True)
    pb_df.to_csv(out_dir)


Processing cell type: neutrophils
Processing cell type: macrophages
Processing cell type: cell_debris
Processing cell type: endothelial
Processing cell type: fibroblasts
Processing cell type: T
Processing cell type: AT2
Processing cell type: AT1
Processing cell type: mast
Processing cell type: B
Processing cell type: alveolar_macrophages
Processing cell type: basal
Processing cell type: plasma
Processing cell type: secretory
Processing cell type: smooth_muscle
Processing cell type: ciliated
